# Copy Validated Files to Legacy R: Drive via Databridge SMB (02b)

Copies validated SWIFT DVR files from the UC Volume archive folder to the legacy NetApp
R: drive drop location using the OEDIS Databridge SMB handler (`oedis.handler.smb.upload`).

This ensures the existing DataStage process can continue to load FMBODW during
the migration transition period.

**Requires classic compute** — serverless cannot reach on-prem NetApp.

Credentials:
- Service account: `bocad\triton_s`
- Key Vault scope: configured via `secret_scope` variable
- UNC path: parameterised per environment in `databricks.yml`

In [0]:
%run ../../common

In [0]:
import json
import logging
from datetime import datetime

# Job parameters. DataBridge connection settings come from the shared job
# context (_ctx) built by common.py, which resolves them from Key Vault.
dbutils.widgets.text("legacy_r_drive_path", "", "Legacy R drive UNC path")
dbutils.widgets.text("valid_files_json", "", "Valid files JSON from notebook 01")

logger = logging.getLogger(__name__)
r_drive_path = dbutils.widgets.get("legacy_r_drive_path").strip()
valid_files_json = dbutils.widgets.get("valid_files_json").strip()

# Fall back to the validate task's value when the job doesn't pass it explicitly.
if not valid_files_json:
    try:
        valid_files_json = dbutils.jobs.taskValues.get(
            taskKey="validate_swift_files", key="valid_files_json", debugValue=""
        ) or ""
    except Exception:
        valid_files_json = ""

valid_files = json.loads(valid_files_json) if valid_files_json else []
file_names = sorted({f["filename"] for f in valid_files if f.get("filename")})


def _exit(status, copied=0, **extra):
    # Lets notify show the copy outcome; a missing value means the copy failed.
    dbutils.jobs.taskValues.set(key="status", value=status)
    dbutils.notebook.exit(json.dumps({
        "copied_count": copied,
        "r_drive_path": r_drive_path,
        "status": status,
        "copied_at": datetime.utcnow().isoformat(),
        **extra,
    }))


if not file_names:
    _exit("NO_VALID_FILES")
if not r_drive_path:
    _exit("SKIPPED_NO_R_DRIVE_PATH", files=file_names)

# Imported only when there is work to do.
from oeds.domain.utils.ingest.ingest_utils import DisConnectionOptions, run_ingest

scope = _ctx["oeds_secrete_scope"]
connection = DisConnectionOptions(
    host=_ctx["internal_databricks_workspace_url"],
    azure_tenant_id=_ctx["azure_tenant_id"],
    azure_client_id=dbutils.secrets.get(scope=scope, key=_ctx["service_spn_id_key"]),
    azure_client_secret=dbutils.secrets.get(scope=scope, key=_ctx["service_spn_secrete_key"]),
    azure_workspace_resource_id=_ctx["internal_databricks_resource_id"],
    dis_job_id=int(_ctx["internal_dis_job_id"]),
)

# Validated files are already in the archive folder (moved there by notebook 01).
archive_root = f'{_ctx["raw_root"]}/archive'
logger.info("Copying %d file(s) from %s to %s", len(file_names), archive_root, r_drive_path)
print(f"Copying {file_names} from {archive_root} to {r_drive_path}")

result = run_ingest(
    connection=connection,
    config_file="default.json",
    template_params={
        "r_drive_path": r_drive_path,
        "archive_root": archive_root,
        "file_names": file_names,
    },
    # SMB destinations can't be listed from Databricks, so skip file verification.
    verification=None,
)

ingest = result.ingest_result
logger.info("DataBridge R drive transfer completed")
_exit(
    "SUBMITTED",
    copied=len(file_names),
    files=file_names,
    databridge_status=str(ingest.get_status()),
    databridge_run_url=str(ingest.get_job_run_page_url()),
)